In [1]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

# === 1. read data ===
df_input = pd.read_excel("ai_infra_clean_us_stock_universe.xlsx")
tickers = df_input["Ticker"].unique().tolist()

# === 2. assign 6M Return time window from today ===
end_date = datetime.today()
start_date = end_date - timedelta(days=180)

# === 3. extra data ===
results = []

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
        info = stock.info

        if hist.empty or len(hist["Close"]) < 2:
            continue

        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        six_month_return = ((end_price - start_price) / start_price) * 100

        pe = info.get("trailingPE")
        roe = info.get("returnOnEquity")
        sector = info.get("sector")

        results.append({
            "Ticker": ticker,
            "Theme": "AI Infrastructure",
            "Driver": df_input[df_input["Ticker"] == ticker]["Driver"].values[0],
            "Sector": sector,
            "Company Name": info.get("longName"),
            "Company Website": info.get("website"),
            "PE": pe,
            "ROE": roe * 100 if roe else None,
            "6M Return": round(six_month_return, 2)
        })

    except Exception as e:
        print(f"⚠️ Failed to fetch {ticker}: {e}")
        continue

# === 4. fill na ===
df_result = pd.DataFrame(results)
df_result["Data Quality"] = "Clean"

for driver in df_result["Driver"].unique():
    subset = df_result[df_result["Driver"] == driver]
    median_pe = subset["PE"].median()
    median_roe = subset["ROE"].median()

    pe_missing = (df_result["Driver"] == driver) & (df_result["PE"].isnull())
    roe_missing = (df_result["Driver"] == driver) & (df_result["ROE"].isnull())

    df_result.loc[pe_missing, "PE"] = median_pe
    df_result.loc[pe_missing, "Data Quality"] = "Estimated PE"

    df_result.loc[roe_missing, "ROE"] = median_roe
    df_result.loc[roe_missing & (df_result["Data Quality"] == "Estimated PE"), "Data Quality"] = "Estimated PE & ROE"
    df_result.loc[roe_missing & (df_result["Data Quality"] == "Clean"), "Data Quality"] = "Estimated ROE"

# === 5. assign 1.0–3.0 based on percentile rank ===
def percentile_score(series, ascending=True):
    ranks = series.rank(pct=True, ascending=ascending)
    return (1.0 + 2.0 * ranks).round(4)

df_result["Value Score"] = 0.0
df_result["Quality Score"] = 0.0
df_result["Momentum Score"] = 0.0

for driver in df_result["Driver"].unique():
    subset = df_result["Driver"] == driver
    df_result.loc[subset, "Value Score"] = percentile_score(df_result.loc[subset, "PE"], ascending=False)
    df_result.loc[subset, "Quality Score"] = percentile_score(df_result.loc[subset, "ROE"], ascending=True)
    df_result.loc[subset, "Momentum Score"] = percentile_score(df_result.loc[subset, "6M Return"], ascending=True)

# === 6. Composite Score calculation ===
df_result["Composite Score"] = (
    df_result["Value Score"] * 0.25 +
    df_result["Quality Score"] * 0.35 +
    df_result["Momentum Score"] * 0.40
).round(4)

# === 7. save Excel ===
df_result.to_excel("ai_infra_fundamental_scored_percentile_theme_sector.xlsx", index=False)
print("✅ Saved to ai_infra_fundamental_scored_percentile_theme_sector.xlsx")


✅ Saved to ai_infra_fundamental_scored_percentile_theme_sector.xlsx
